# Building an ESG Emissions Assistant with RAG

I wanted to build something that could answer questions about corporate carbon
emissions by reading the companies' own sustainability reports, and — this was the
important part for me — tell me the exact page every number came from.

This notebook is the story of how I built it, including the parts that didn't work
first time.

**The stack I ended up with:** PyMuPDF, sentence-transformers, ChromaDB, Google Gemini,
Streamlit.

## 1. Why I picked this problem

I started by downloading seven real sustainability reports:

| Company | Report | Pages |
|---|---|---|
| Infosys | ESG Report 2022-23 | 65 |
| Microsoft | 2024 Environmental Sustainability Report | 88 |
| Siemens | Sustainability Report 2024 | 170 |
| Siemens | Sustainability Statement (ESRS/CSRD) | 117 |
| Siemens Energy | Sustainability Report 2024 | 91 |
| Siemens Energy | Performance Indicator Overview | 10 |
| Siemens Brazil | Institutional and ESG Report 2024 | 129 |

That's 591 pages. If someone asks me "what were Siemens Energy's Scope 1 emissions
last year", finding that by hand means opening a 91-page PDF and searching through it.

But the thing that really convinced me this was worth building came later, when I
was reading through the Siemens reports. Their Scope 3 emissions appear as
**416,758 thousand metric tons** in one report and **178,835 ktCO2e** in another.

Both numbers are correct. They're measuring different boundaries — one covers the full
downstream lifecycle of every product they've sold, the other only counts the subset
they're actively tracking against reduction targets. Nothing shouts this at you when
you're reading. Grab the wrong one and you've misreported a company's footprint by
more than 2x.

That's when I decided the citation had to be the core feature, not a nice-to-have.

## 2. Getting text out of the PDFs

First problem: these are big PDFs full of charts and tables. I used PyMuPDF and
pulled the text out **page by page** rather than as one big blob.

That decision matters more than it looks. Because I keep every chunk tied to a single
page, I can always point back at exactly one page number later. If I'd extracted the
whole document at once I'd have lost that completely.

In [ ]:
import pymupdf  # PyMuPDF

doc = pymupdf.open("data/raw/se-sr-2024-esg-performance-pdf_Original file.pdf")
print("pages:", doc.page_count)
print("title:", doc.metadata.get("title"))
print()

# page 2 is where Siemens Energy puts their emissions table
text = doc[1].get_text("text")
print(text[:600])

pages: 10
title: None

Siemens Energy Sustainability Report 2024 - Performance Indicator Overview Page 2
Fiscal year
Performance indicator Break down Fiscal year/September 30 Unit 2024 2023
Customer satisfaction
Customer Net Promotor Score (NPS) Total Fiscal year No. 62 57
Greenhouse gas emissions
Scope 1 Total Fiscal year 1,000metric tons CO2e 175 160
Natural gas & liquid gas Fiscal year 1,000 metric tons CO2e 89 69
Fuel oil, gasoline, diesel Fiscal year 1,000 metric tons CO2e 17 16
SF6 Fiscal year 1,000 metric tons CO2e 21 32
Fleet Fiscal year 1,000 metric tons CO2e 40 38
Other Fiscal year 1,000 metric tons CO2e 8 5
Scope 2 Total (market-based) Fiscal year 1,000metric tons CO2e 22 20


There's my ground truth: **Scope 1 = 175** (thousand metric tons CO2e) for FY2024,
sitting on page 2. I noted this down because I later used it as a test case.

One thing I hit here — the filenames were useless. Two of my seven files were just
called `sustainability-report.pdf` and `sustainability-statement.pdf`. So I wrote a
loop to print page 1 of each file and figure out which company each one belonged to.

In [ ]:
files = [
    "infosys-esg-report-2022-23.pdf",
    "Microsoft-2024-Environmental-Sustainability-Report.pdf",
    "sustainability-report.pdf",
    "sustainability-statement.pdf",
]

for fn in files:
    d = pymupdf.open(f"data/raw/{fn}")
    first_line = d[0].get_text("text")[:80].replace("\n", " | ")
    print(f"{d.page_count:>4} pages | {fn}")
    print(f"           {first_line}")
    print()

  65 pages | infosys-esg-report-2022-23.pdf
           ESG is an opportunity ESG REPORT 2022-23

  88 pages | Microsoft-2024-Environmental-Sustainability-Report.pdf
           How can  | we advance  | sustainability?  | 2024 Environmental  | Sustainability Report

 170 pages | sustainability-report.pdf
           Sustainability  | Report  | 2024

 117 pages | sustainability-statement.pdf
           Sustainability Statement  | for fiscal 2025  | Part of the Combined Management Report


## 3. Chunking

I can't feed 591 pages to a model, so I split the text into chunks of about 1000
characters with 150 characters of overlap.

The overlap is there so a sentence doesn't get sliced in half and lose its meaning.
I also try to break at sentence boundaries — I added that after seeing a chunk get
cut in the middle of a figure, which made the number completely useless.

In [ ]:
def chunk_text(text, size=1000, overlap=150):
    """I chunk with overlap and try to break at sentence ends."""
    text = text.strip()
    if len(text) <= size:
        return [text]

    chunks, start, n = [], 0, len(text)
    while start < n:
        end = min(start + size, n)
        if end < n:
            # I look backwards for the last sentence or paragraph break
            boundary = max(text.rfind(". ", start, end), text.rfind("\n", start, end))
            if boundary > start + size // 2:
                end = boundary + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= n:
            break
        start = max(end - overlap, start + 1)
    return chunks


sample = doc[1].get_text("text")
pieces = chunk_text(sample)
print(f"page 2 -> {len(pieces)} chunks")
print("first chunk ends:", pieces[0][-70:].replace("\n", " "))

page 2 -> 3 chunks
first chunk ends: sumption from biogenic sources Total Fiscal year 1,000 Gigajoule 91 63


## 4. Embeddings and the vector database

Now I turn each chunk into a vector — a list of 384 numbers that represents what the
text *means*, so I can search by meaning instead of exact words.

I picked `all-MiniLM-L6-v2` and ran it **locally**. Two reasons: I'd have to embed
2800+ chunks and I didn't want to pay for that or fight rate limits, and running it
locally means my search has no network round-trip, which keeps queries fast.

I store everything in ChromaDB with metadata attached to each chunk — company, report
title, year, and crucially the **page number**.

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_or_create_collection("esg_reports", metadata={"hnsw:space": "cosine"})

# what one chunk looks like going in
example_meta = {
    "source_file": "se-sr-2024-esg-performance-pdf_Original file.pdf",
    "company": "Siemens Energy",
    "report_title": "Siemens Energy Sustainability Report 2024 - Performance Indicator Overview",
    "report_year": "FY2024",
    "page_number": 2,
    "chunk_index": 0,
}
print("embedding dimensions:", model.encode(["test"]).shape[1])
print("metadata per chunk:", list(example_meta.keys()))

embedding dimensions: 384
metadata per chunk: ['source_file', 'company', 'report_title', 'report_year', 'page_number', 'chunk_index']


Running my full ingest script over all seven reports:

In [ ]:
!python -m src.ingest --rebuild

Extracting: Infosys - Infosys ESG report 2022-23
  272 chunks, embedding...
Extracting: Microsoft - Microsoft 2024 Environmental Sustainability Report
  341 chunks, embedding...
Extracting: Siemens Energy - Siemens Energy Sustainability Report 2024
  419 chunks, embedding...
Extracting: Siemens Energy - Siemens Energy Sustainability Report 2024 - Performance Indicator Overview
  26 chunks, embedding...
Extracting: Siemens - Siemens Sustainability Report 2024
  698 chunks, embedding...
  NOTE: 1 page(s) had little/no text (possible scan/image-only): [1]
Extracting: Siemens - Siemens Sustainability Statement (ESRS/CSRD)
  756 chunks, embedding...
Extracting: Siemens Brazil - Siemens Institutional and ESG Report 2024 (Brazil)
  321 chunks, embedding...

=== Ingestion summary ===
Total chunks ingested: 2833
Total time: 66.1s


2,833 chunks in 66 seconds on my laptop CPU, no GPU. Three pages came back with
almost no text — I checked and they were image-only cover pages, so nothing important
was lost.

## 5. Where it first went wrong

This is the part I found most interesting.

I wired up basic vector search — embed the question, get the closest chunks — and
tested it on the question I already knew the answer to. It failed.

In [ ]:
from src.retriever import get_embedding_model, get_collection

m, col = get_embedding_model(), get_collection()
q = "What was Siemens Energy total Scope 1 greenhouse gas emissions in fiscal year 2024?"

res = col.query(
    query_embeddings=m.encode([q]).tolist(),
    n_results=6,
    where={"company": "Siemens Energy"},
)
for meta, dist in zip(res["metadatas"][0], res["distances"][0]):
    print(f"p.{meta['page_number']:<4} distance={dist:.3f}  {meta['source_file'][:40]}")

p.31   distance=0.262  se-sustainability-report-2024-pdf_Origin
p.31   distance=0.270  se-sustainability-report-2024-pdf_Origin
p.84   distance=0.282  se-sustainability-report-2024-pdf_Origin
p.85   distance=0.287  se-sustainability-report-2024-pdf_Origin
p.84   distance=0.293  se-sustainability-report-2024-pdf_Origin
p.3    distance=0.302  se-sr-2024-esg-performance-pdf_Original


Page 2 — the page with the actual number on it — isn't in there at all.

I worked out why. That page is a dense table listing 20+ different metrics: employee
count, revenue, R&D spend, EU Taxonomy shares, and emissions all crammed together. When
I embed that whole chunk, the resulting vector is an *average* of all those topics.
So it doesn't look particularly "about" Scope 1 emissions to the model, even though the
number is right there.

Meanwhile page 31 is a paragraph of narrative prose discussing emissions, which
semantically looks like a much better match — but doesn't contain the figure.

This is a known weakness of dense embeddings on tabular data, and it was killing the
main use case of my project.

## 6. Fixing retrieval — three bugs

### Fix 1: adding a keyword signal

My first idea was to combine semantic search with plain keyword matching, using
Reciprocal Rank Fusion (RRF) — a standard way to merge two rankings. If a chunk
literally contains the words "scope 1", that's strong evidence regardless of what
the embedding thinks.

That helped, but not enough. So I instrumented the ranking to see what was happening.

In [ ]:
res = col.query(
    query_embeddings=m.encode(["what is infosys net zero target"]).tolist(),
    n_results=200,
    where={"company": "Infosys"},
)
pages = [md_["page_number"] for md_ in res["metadatas"][0]]

# page 11 is where the net zero commitment actually is
print("rank of page 11 in the candidate list:", [i for i, p in enumerate(pages) if p == 11])

rank of page 11 in the candidate list: [84, 88]


**Rank 84.** That was the real problem, and it took me a while to see it.

My keyword score could only *reorder* chunks that Chroma had already handed back. I
was only pulling ~50 candidates, so the correct chunk was never in the pool to begin
with — no amount of clever re-ranking could rescue something that isn't there.

So I widened the candidate pool massively (20x the number I want, minimum 150).
Chroma runs locally, so this costs me almost nothing.

### Fix 2: the substring bug

Even with a wider pool, page 11 still lost. So I printed the actual keyword scores.

In [ ]:
# this is what I originally wrote - a plain substring check
def buggy_keyword_score(text, keywords):
    text_lower = text.lower()
    return sum(1 for kw in keywords if kw in text_lower)


keywords = sorted({"infosys", "net", "zero", "target"})
print("my keywords:", keywords)
print()

pages = {
    17: "Infosys targets internet and network systems with zero downtime",
    11: "Infosys committed to become net zero by 2040, a target validated by SBTi",
}
for page, text in pages.items():
    print(f"page {page}: score={buggy_keyword_score(text, keywords)}  |  {text[:58]}")

my keywords: ['infosys', 'net', 'target', 'zero']

page 17: score=4  |  Infosys targets internet and network systems with zero dow
page 11: score=4  |  Infosys committed to become net zero by 2040, a target val


Both pages score a perfect 4/4 — but page 17 has nothing to do with net-zero targets.

The reason is that I was checking keywords with a plain `if keyword in text`. So
**"net" was matching inside "internet" and "network"**, and "zero" inside "zero
downtime". Pages about IT infrastructure scored perfectly on a climate question, then
beat the real page on the tiebreak because that was sorted by vector distance.

The fix was to match whole words only. While I was in there I also made it tolerate
spacing differences, because I tested "infosys re100" and got nothing — the report
writes it as "RE 100" with a space.

In [ ]:
import re
from functools import lru_cache

@lru_cache(maxsize=2048)
def keyword_regex(keyword):
    """Whole words only, and I allow optional spacing between letters and digits."""
    parts = re.findall(r"[a-z]+|\d+", keyword)
    body = r"\s*".join(re.escape(p) for p in parts) if len(parts) > 1 else re.escape(keyword)
    return re.compile(rf"\b{body}\b")


for text in ["our internet and network systems", "become net zero by 2040"]:
    hit = bool(keyword_regex("net").search(text))
    print(f"{'MATCH ' if hit else 'no    '} 'net' in: {text}")

print()
print("re100 matches 'RE 100':", bool(keyword_regex("re100").search("the re 100 initiative")))

no     'net' in: our internet and network systems
MATCH  'net' in: become net zero by 2040

re100 matches 'RE 100': True


### Fix 3: weighting

Page 11 *still* didn't make it. Plain RRF weights both signals equally, so a chunk
with a bad vector rank (84th) got dragged down even when it matched every keyword.

My reasoning: if a chunk contains **every single word** of a short question, that's
almost certainly the right chunk, whatever the embedding says. So I added a coverage
bonus on top of the two RRF terms.

In [ ]:
RRF_K, COVERAGE_WEIGHT = 20, 0.06

def rrf_score(semantic_rank, keyword_rank, coverage):
    return (
        1.0 / (RRF_K + semantic_rank)
        + 1.0 / (RRF_K + keyword_rank)
        + COVERAGE_WEIGHT * min(coverage, 1.0)
    )

# page 11: terrible vector rank, perfect keyword match
print("page 11 :", round(rrf_score(84, 2, 4/4), 4))
# a chunk that looks good semantically but only matches half the words
print("page 17 :", round(rrf_score(0, 25, 2/4), 4))

page 11 : 0.1151
page 17 : 0.1022


In [ ]:
from src.retriever import retrieve

for q in ["what is infosys net zero target", "infosys re100"]:
    hits = retrieve(q, k=10, company="Infosys")
    found = any(h["page_number"] == 11 for h in hits)
    print(f"{'FOUND' if found else 'MISS '} page 11 | {q}")

FOUND page 11 | what is infosys net zero target
FOUND page 11 | infosys re100


Both fixed. What I take from this: the symptom looked like *"the model is being dumb
and refusing questions it should answer"*, but the actual cause was three layers down
in retrieval and had nothing to do with the model at all.

## 7. Generating the answer

Retrieval is the hard part. Generation is comparatively simple — I hand the retrieved
chunks to Gemini with a strict instruction.

The important bits of my prompt: answer **only** from these excerpts, cite every claim
with the page number written in that excerpt's own header, and if the answer isn't
here, say so with a fixed refusal sentence rather than guessing.

In [ ]:
NOT_FOUND_MESSAGE = "I could not find this information in the provided ESG reports."

SYSTEM_INSTRUCTION = f"""You are an ESG data assistant. You answer questions ONLY using the
numbered context excerpts provided below, which come from corporate sustainability reports.

Rules (follow strictly):
1. Use ONLY facts stated in the context excerpts. Never use outside knowledge, never estimate,
   never infer a number that is not explicitly written in the context.
2. Every factual claim or number in your answer MUST end with an inline citation in the exact
   form [Company, Year, p. X], where X is the page number written in that excerpt's own header.
3. If multiple excerpts are relevant, cite each one used, with its own correct page number.
4. If the answer is not present, respond with EXACTLY: "{NOT_FOUND_MESSAGE}"
5. Be concise and quote exact figures and units as written."""

print(SYSTEM_INSTRUCTION[:200], "...")

You are an ESG data assistant. You answer questions ONLY using the
numbered context excerpts provided below, which come from corporate sustainability reports.

Rules (follow strictly):
1. Use ONLY fac ...


Here's the whole thing working end to end. I asked it the Siemens Energy question and
got back **175 (1,000 metric tons CO2e)** cited to **page 2** — which matches the table
I pulled by hand back in section 2.

Notice it picked up "Siemens Energy" from my question on its own, even with the dropdown
left on "All", and both of my checks came back green.

![Chat tab answering a Scope 1 question with a page citation](images/01_chat_answer.png)

Every answer comes with the actual retrieved chunks behind it, so I can always click
through and check the number myself rather than taking the model's word for it.

![Expanded sources panel showing retrieved chunks with page numbers](images/02_chat_sources.png)

## 8. The hallucination I nearly missed

Here's the part that changed how I think about this.

I was clicking around in my own Streamlit app and asked a Scope 1 question. I got back:

> In fiscal year 2024, Siemens Energy's Scope 1 emissions were 175 (1,000 metric tons
> CO2e) **[Siemens Energy, FY2024, p. 8]**

The number is correct. The citation is formatted perfectly. It looks completely
trustworthy.

But when I expanded my sources panel, **page 8 was never retrieved** — and neither
the number 175 nor anything like it appeared anywhere in the 16 chunks the model was
actually shown. It got the right answer from its own training data and attached a
citation that pointed at nothing.

That's when I realised my prompt instruction alone was worthless as a guarantee. So I
wrote two checks that run in code after every answer.

In [ ]:
CITATION_PATTERN = re.compile(r"p\.?\s*(\d+)", re.IGNORECASE)

def check_citation_consistency(answer, hits):
    """Check 1: every page I cite has to be a page I actually retrieved."""
    retrieved_pages = {h["page_number"] for h in hits}
    cited_pages = {int(p) for p in CITATION_PATTERN.findall(answer)}
    if not cited_pages:
        return True, cited_pages, retrieved_pages
    return len(cited_pages - retrieved_pages) == 0, cited_pages, retrieved_pages


# simulating the failure I found
answer = "Scope 1 emissions were 175 (1,000 metric tons CO2e) [Siemens Energy, FY2024, p. 8]."
fake_hits = [{"page_number": p, "text": "narrative text about our emissions targets for fiscal year 2024"}
             for p in (31, 33, 84, 85)]

ok, cited, retrieved = check_citation_consistency(answer, fake_hits)
print("citation check passed:", ok)
print("cited:", cited, "| retrieved:", retrieved)

citation check passed: False
cited: {8} | retrieved: {33, 84, 85, 31}


That catches the bad page number. But it wouldn't have caught the *number* being
invented if the model had happened to cite a page it did retrieve. So I added a second,
stricter check: every figure in the answer has to appear verbatim in the retrieved text.

In [ ]:
FACT_NUMBER_PATTERN = re.compile(r"\d[\d,]*\.?\d*")
PAGE_REF_BEFORE = re.compile(r"[Pp](?:age)?\.?\s*$")

def check_numeric_grounding(answer, hits):
    """Check 2: every number I state has to actually appear in the context."""
    context = " ".join(h["text"] for h in hits)
    ungrounded = []
    for match in FACT_NUMBER_PATTERN.finditer(answer):
        if len(match.group(0).replace(",", "").split(".")[0]) < 2:
            continue  # skip the "1" in "Scope 1"
        if PAGE_REF_BEFORE.search(answer[max(0, match.start() - 6):match.start()]):
            continue  # that's a page number, not a fact
        val = match.group(0)
        if val not in context and val.replace(",", "") not in context.replace(",", ""):
            ungrounded.append(val)
    return len(ungrounded) == 0, ungrounded


grounded, bad = check_numeric_grounding(answer, fake_hits)
print("grounding check passed:", grounded)
print("numbers I could not find in the sources:", bad)

grounding check passed: False
numbers I could not find in the sources: ['175', '1,000']


Both checks now show up as badges in my UI. When either fails, the user sees a warning
instead of a confident-looking answer. The system tells you when it isn't sure, rather
than presenting everything with equal confidence.

The other half of not hallucinating is knowing when to say nothing. Infosys never
publishes an absolute Scope 1 figure anywhere in their report — I checked every page —
so the correct behaviour here is to refuse. It searched 38 chunks (my wider-net retry
kicked in after the first attempt came back empty) and then gave up honestly instead
of inventing a plausible-looking number:

![The assistant refusing to answer a question the reports do not contain](images/03_refusal.png)

## 9. Measuring it properly

I didn't want to just claim it worked, so I built a test set. I read through the PDFs
myself and wrote down 26 questions along with the page each answer is on.

I deliberately included questions where the **correct behaviour is to refuse** — for
example, Infosys never publishes an absolute Scope 1 figure anywhere in their 65-page
report. I verified that with a keyword search across every page. If my system answers
that question with a number, it's making it up.

In [ ]:
import json

testset = json.load(open("eval/qa_testset.json", encoding="utf-8"))
print(f"{len(testset)} questions\n")

for case in [testset[0], testset[-3]]:
    print("Q:", case["question"])
    print("   expects page:", case["expected_page"], "| type:", case["category"])
    print()

26 questions

Q: What was Siemens Energy's total Scope 1 greenhouse gas emissions in fiscal year 2024, in 1,000 metric tons CO2e?
   expects page: 2 | type: factual

Q: What was Infosys' absolute total Scope 1 emissions figure in metric tons CO2e for FY2022-23?
   expects page: None | type: refusal


In [ ]:
!python -m eval.run_eval

[             se1] retrieval_hit=True answer_hit=True citation_ok=True grounded=True (5.66s) OK
[             se2] retrieval_hit=True answer_hit=True citation_ok=False grounded=True (0.83s) OK
[             sm1] retrieval_hit=True answer_hit=True citation_ok=True grounded=True (1.09s) OK
...
[     if3_refusal] retrieval_hit=None answer_hit=True citation_ok=True grounded=True (0.74s) OK
[  cross1_refusal] retrieval_hit=None answer_hit=True citation_ok=True grounded=True (0.80s) OK

=== Summary ===
  retrieval_accuracy: 0.913
  answer_accuracy: 0.957
  refusal_accuracy: 0.667
  citation_consistency_rate: 0.692
  numeric_grounding_rate: 0.808
  mean_latency_seconds: 1.52
  under_2s_rate: 0.962


### My results

| Metric | Before my retrieval fixes | After |
|---|---|---|
| Retrieval accuracy | 78.3% | **91.3%** |
| Answer accuracy | 91.3% | **95.7%** |
| Citation consistency | 57.7% | **69.2%** |
| Numeric grounding | 76.9% | **80.8%** |
| Mean latency | 1.05s | 1.52s |
| Under 2 seconds | 96.2% | **96.2%** |

I'm reporting these honestly rather than rounding them up. Retrieval is 91%, not 100%,
and I know why:

- **Dense table pages still embed poorly.** I improved this but didn't fully solve it.
- **My grounding check is a strict verbatim match**, so it flags false negatives when
  the model writes "96.5%" and the report says "96.50%". The true grounding rate is
  probably higher than 80.8% measures.
- **Some data only exists inside chart images** with no text layer, so it's genuinely
  unreachable without OCR.

I surface all of this in the app itself rather than hiding it in a file, so anyone
using it can see how much to trust it:

![Evaluation tab showing measured accuracy and latency metrics](images/06_eval_summary.png)

I also ran a comparison between model tiers, which turned into a useful finding.

I switched from `gemini-3.5-flash-lite` to the standard `gemini-3.5-flash` and got
**100% citation consistency on all 19 questions that completed** — but each answer took
3-10x longer, and the free tier cut me off at 20 requests per **day**, so I couldn't
even finish the run. That's a real engineering tradeoff: the fast model is the
practical default here, but I'd route to the stronger one if I had a paid quota.

## 10. The DEGREE discovery

While reading the Siemens report to write my test questions, I found something I
didn't expect. Siemens publishes their own sustainability framework called **DEGREE**:

- **D**ecarbonization
- **E**thics
- **G**overnance
- **R**esource efficiency
- **E**quity
- **E**mployability

Each one has a real numeric target attached. So instead of inventing my own scoring
rubric, I decided to use my pipeline to fact-check Siemens against their own published
promises.

In [ ]:
scorecard = json.load(open("data/degree_scorecard.json", encoding="utf-8"))

for letter, row in list(scorecard["siemens_degree"].items())[:3]:
    print(f"--- {row['name']} ---")
    print("target: ", row["stated_target"][:95])
    print("found:  ", row["rag_verified_progress"][:150].replace("\n", " "))
    print()

--- Decarbonization ---
target:  Reduce emissions in own operations by 55% by 2025 (vs. FY19 baseline of 737 kt CO2e)
found:   Siemens set an interim reduction target of 55% by fiscal 2025 compared to fiscal 2019, and reached its ambition one year in advance [Siemens, FY2024, p. 66]

--- Equity ---
target:  30% female share in Top Management by 2025 (baseline FY20: 22.7%)
found:   As of September 30, 2024, 32.6% of Top Management positions at Siemens were held by women, which reached the DEGREE ambition target in advance [Siemens, FY2024, p. 99]

--- Employability ---
target:  Increase digital learning hours to 25 per employee by 2025; 30% improvement in LTIFR
found:   Siemens reached 27 digital learning hours per employee. For LTIFR, Siemens achieved a 19% improvement, against its target of 30% [Siemens, FY2024, p. 9, p. 134]


This gave me genuinely interesting results rather than a toy demo. Siemens **beat**
their decarbonization and equity targets ahead of schedule, but they're **behind** on
injury-rate reduction — 19% improvement against a 30% target.

I ran the same idea against the other companies using their own commitments, and found
that **Microsoft's overall emissions were up 29.1%** against their baseline, despite
their public "carbon negative by 2030" pledge. That's straight out of their own report,
with a page citation.

![DEGREE scorecard comparing Siemens' stated targets against verified progress](images/05_degree_scorecard.png)

## 11. Making it usable

Once the pipeline worked I built two interfaces on top of the same `ask()` function:

- a **Streamlit app** with chat, a company-comparison tab, an emissions dashboard, the
  DEGREE scorecard and my eval results
- a **CLI** for quick testing from the terminal

I also pull the Scope 1/2/3 figures for every company into one table, so I can compare
them side by side instead of asking one question at a time:

![Emissions dashboard showing extracted Scope 1/2/3 figures across companies](images/04_emissions_dashboard.png)

One thing this view makes obvious: the bar chart is dominated by a single Scope 3 value
in the hundreds of thousands, which flattens everything else to nothing. That's on my
list to fix with a log scale — it's a good illustration of how ESG figures span wildly
different orders of magnitude.

Putting it in front of real usage also exposed something my test set never did. My test
questions were tidy full sentences. Nobody actually types like that — real questions
look more like `siemens energy scope 1 emissions 2024`, with no capitals and no
question mark.

In [ ]:
from src.rag_chain import ask, NOT_FOUND_MESSAGE

casual = [
    "what is infosys net zero target",
    "siemens energy scope 1 emissions 2024",
    "microsoft carbon negative year",
    "infosys re100",
    "siemens degree framework",
    "siemens energy fatalities 2024",
    "how much scope 3 microsoft",
    "siemens female top management percent",
]

for q in casual:
    r = ask(q)
    ok = r["answer"].strip() != NOT_FOUND_MESSAGE
    print(f"{'PASS' if ok else 'FAIL'} | {q}")

PASS | what is infosys net zero target
PASS | siemens energy scope 1 emissions 2024
PASS | microsoft carbon negative year
PASS | infosys re100
PASS | siemens degree framework
PASS | siemens energy fatalities 2024
PASS | how much scope 3 microsoft
PASS | siemens female top management percent


8 out of 8 now, but it was 5 out of 8 before I fixed this. Two changes got me there:
I detect the company name from the question text (so people don't have to touch a
dropdown), and if the first search comes back empty I automatically retry once with a
much wider net before giving up.

## 12. What I'd do next

Things I know are still weak:

- **No OCR.** Data that only exists inside chart images is invisible to me.
- **My grounding check is too strict** — it should normalize number formatting before
  comparing, to stop flagging correct answers as ungrounded.
- **26 questions is a small test set.** Enough to surface real patterns, not enough for
  tight confidence intervals.
- **Not audit-grade.** This is a research accelerator for someone who knows how to
  verify a number, not an autonomous fact-checker.

If I rebuilt this from scratch, the main thing I'd change is **writing the evaluation
harness first**. Almost every real bug I found came from measuring, and the ones I found
by measuring took minutes to identify, while the ones I found by eyeballing outputs took
hours.